<a href="https://colab.research.google.com/github/adamsi/API-Assistant-MultiAgent/blob/main/image_classification_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image classification Flow


1. Download images from kaggle to root data directory
2. Define transforms pipeline, Create full_dataset with ImageFolder(...),
random_split to: test_ds, train_ds.
Create dataLoaders (model input)
3. Define model architucture:
transfer learning of pretrained resnet101,
freeze base layers, replace head and train new head,
optionally fine-tune deeper layers (not here)

4. Create train_step and test_step functions (pass dataLoader and model)
5. Main train function, pesudo:  

```
for epoch in eopchs:
  train_step(...)
  test_step(...)
  print_loss_and_accuracy(...)
```


In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split
import torchvision
from torchvision import transforms, datasets
import torch
from torch.utils.data import DataLoader
from torch import nn
from pathlib import Path
import random
from PIL import Image
import torch
from torchvision import transforms
from torch import nn
import matplotlib.pyplot as plt
import numpy as np
from torchvision.transforms import InterpolationMode

In [ ]:
#TODO download from kaggle
weights = torchvision.models.ResNet101_Weights.DEFAULT
transforms = weights.transforms()

In [ ]:
def create_dataloaders(data_dir:str,
                       transform:transforms:Compose,
                       batch_size:int,
                       num_workers:int=0):
  full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
  train_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
  )

  class_names = full_dataset.classes

  train_dataloader = DataLoader(
      train_dataset,
      batch_size,
      True,
      num_workers=num_workers,
      pin_memory=True
  )

  test_dataloader = DataLoader(
      test_dataset,
      batch_size,
      True,
      num_workers=num_workers,
      pin_memory=True
  )

  return train_dataloader, test_dataloader, class_names

In [ ]:
def init_model_architucture(class_names:list[str]):
  device = "cuda" if torch.cuda.is_available() else "cpu"
  model = torchvision.model.resnet101(weights=weights).to(device)

  for param in model.parameters():
    param.requries_grad=False

  model.fc = nn.Linear(in_features=2048, out_features=class_names, bias = True).to(device)

  for param in model.fc.parameters():
    param.requires_grad=True

  loss_fn = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

  return model, loss_fn, optimizer, device

In [1]:
def train_step(model:torch.nn.Module,
               loss_fn:torch.nn.Module,
               optimizer:torch.optim.Optimizer,
               data_loader:torch.utils.data.DataLoader,
               device:torch.device):
  model.train()
  train_loss=0
  train_acc=0

  for (x, y) in data_loader:
    x, y = x.to(device), y.to(device)
    y_logits = model(x)
    loss = loss_fn(y_logits, y)
    train_loss += loss.item()

    optimizer.zero_grad()
    loss_dn.backward()
    optimizer.step()

    train_acc += (torch.argmax(torch.softmax(y_logits,dim=1), dim=1) == y).sum().item() / len(y_logits)

  train_loss /= len(data_loader)
  train_acc /= len(data_loader)

  return train_loss, train_acc



def test_step(model:torch.nn.Module,
               loss_fn:torch.nn.Module,
               data_loader:torch.utils.data.DataLoader, device:torch.device):

  model.eval()
  test_loss = 0
  test_acc = 0

  with torch.inference_mode():
    for (x,y) in data_loader:
      x, y = x.to(device), y.to(device)

      y_logits = model(x)
      loss = loss_fn(y_logits, y)
      test_loss += loss.item()

      test_acc += (torch.argmax(torch.softmax(y_logits,dim=1), dim=1) == y).sum().item() / len(y_logits)

  test_loss /= len(data_loader)
  test_acc /= len(data_loader)

  return test_loss, test_acc



SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (1530286825.py, line 7)

In [ ]:
def train(model:torch.nn.Module,
          train_dl:torch.utils.data.DataLoader,
          test_dl:torch.utils.data.DataLoader,
          loss_fn:torch.nn.Module,
          optimizer:torch.optim.Optimizer,
          device:torch.device,
          epochs:int=100):

  for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_step(model=model,loss_fn=loss_fn,optimizer=optimizer,
                                         data_loader=train_dataloader, device=device)
    test_loss, test_acc = test_step(model=model,loss_fn=loss_fn,
                                         data_loader=test_dataloader, device=device)
    print(f"Epoch: {epoch} |"
            f"train_loss: {train_loss} |"
            f"train_acc: {train_acc} |"
            f"test_loss: {test_loss} |"
            f"test_acc: {test_acc} |")